In [3]:
import sys
import os
import torch
from thop import profile

# 1. Append parent directory so Python can find the 'src' folder
sys.path.append(os.path.abspath('..'))

# 2. Import your single dynamic class
from src.models import TriNetModel

def analyze_model_complexity(model, input_size=(1, 3, 224, 224)):
    """Profiles a PyTorch model to calculate GFLOPS and Total Parameters."""
    model.eval()
    
    # Create a dummy tensor matching your WiFi CSI heatmap dimensions
    dummy_input = torch.randn(input_size)
    
    # thop calculates MACs (Multiply-Accumulates). 1 MAC roughly equals 2 FLOPs.
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    flops = macs * 2
    
    # Convert to GigaFLOPs and Millions for readability
    gflops = flops / 1e9
    params_millions = params / 1e6
    
    return gflops, params_millions

# 3. Instantiate the architectures by passing your specific backbone names
# (pretrained=False is used because we just need the structural math, not the downloaded weights)
models_to_test = {
    "EfficientNet-B0": TriNetModel(backbone_name='efficientnet_b0', pretrained=False),
    "DenseNet121": TriNetModel(backbone_name='densenet121', pretrained=False),
    "MobileNetV3": TriNetModel(backbone_name='mobilenet_v3', pretrained=False)
}

# 4. Execute and print formatted table
print(f"{'Model Architecture':<20} | {'GFLOPS':<10} | {'Total Parameters'}")
print("-" * 55)

for name, model in models_to_test.items():
    gflops, params_m = analyze_model_complexity(model)
    print(f"{name:<20} | {gflops:<10.3f} | {params_m:.2f} M")

Model Architecture   | GFLOPS     | Total Parameters
-------------------------------------------------------
EfficientNet-B0      | 0.829      | 4.73 M
DenseNet121          | 5.793      | 7.55 M
MobileNetV3          | 0.122      | 1.29 M
